# 서울시 상권분석 EDA

한국어 컬럼으로 변환된 데이터를 불러옵니다.

In [1]:
import pandas as pd

## 한국어 데이터 불러오기

In [4]:
DATA_DIR = "../data/korean"

DATA_NAMES = [
    "trade_area",
    "stores",
    "sales",
    "floating_population",
    "resident_population",
    "working_population",
    "facilities",
]

datasets = {
    name: pd.read_csv(f"{DATA_DIR}/{name}.csv")
    for name in DATA_NAMES
}

datasets.keys()

dict_keys(['trade_area', 'stores', 'sales', 'floating_population', 'resident_population', 'working_population', 'facilities'])

In [5]:
datasets["trade_area"].head()

,상권_구분_코드,상권_구분_코드명,상권_코드,상권_코드명,X_좌표_EPSG5181,Y_좌표_EPSG5181,자치구_코드,자치구_명,행정동_코드,행정동_명,상권_영역_면적_제곱미터,원본_데이터셋_ID,수집_일시,원본_기준_일자,스키마_버전
0,A,골목상권,3110055,황학동벼룩시장,201642,452260,11140,중구,11140670,황학동,27575,OA-15560,2026-08-11T12:21:04+09:00,2023-06,0.1.0
1,A,골목상권,3110008,배화여자대학교(박노수미술관),197093,453418,11110,종로구,11110515,청운효자동,149264,OA-15560,2026-08-11T12:21:04+09:00,2023-06,0.1.0
2,A,골목상권,3110009,자하문터널,196991,455057,11110,종로구,11110550,부암동,178306,OA-15560,2026-08-11T12:21:04+09:00,2023-06,0.1.0
3,A,골목상권,3110010,평창동서측,197064,456643,11110,종로구,11110560,평창동,369415,OA-15560,2026-08-11T12:21:04+09:00,2023-06,0.1.0
4,A,골목상권,3110017,정독도서관,198581,453781,11110,종로구,11110600,가회동,83855,OA-15560,2026-08-11T12:21:04+09:00,2023-06,0.1.0


## 1. 데이터셋 크기 확인

각 데이터의 행 수와 열 수를 확인한다. 행 수가 0이거나 예상한 컬럼 수와 다르면 수집 범위와 저장 과정의 확인이 필요하다. 데이터마다 분석 단위가 다르므로 행 수가 서로 같을 필요는 없다.

In [6]:
dataset_sizes = pd.DataFrame([
    {"데이터": name, "행 수": df.shape[0], "열 수": df.shape[1]}
    for name, df in datasets.items()
])

dataset_sizes

,데이터,행 수,열 수
0,trade_area,1650,15
1,stores,1604844,18
2,sales,460329,59
3,floating_population,34633,31
4,resident_population,34275,33
5,working_population,34386,30
6,facilities,33138,29


## 2. 결측값 확인

결측값이 있는 데이터와 컬럼, 개수, 비율을 확인한다. 핵심 키의 결측은 데이터 식별이 불가능하므로 문제로 판단한다. 일반 지표와 관리 컬럼의 결측은 컬럼의 의미와 수집 방식을 확인한 뒤 판단한다.

In [7]:
missing_list = []

for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    for column, count in missing.items():
        missing_rate = count / len(df) * 100
        missing_list.append([name, column, count, missing_rate])

missing_df = pd.DataFrame(
    missing_list,
    columns=["데이터", "컬럼", "결측값 수", "결측률 (%)"],
)

missing_df["결측률 (%)"] = missing_df["결측률 (%)"].round(2)

missing_df

,데이터,컬럼,결측값 수,결측률 (%)


## 3. 중복 행 확인

모든 컬럼 값이 완전히 동일한 행을 확인한다. `중복 행 수`가 0이면 정상이다. 0보다 크면 같은 데이터가 반복 저장된 것인지 확인하되, 첫 번째 행은 원본으로 보고 이후 반복된 행만 중복으로 계산한다.

In [8]:
duplicate_list = []

for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    duplicate_rate = duplicate_count / len(df) * 100

    duplicate_list.append([name, duplicate_count, duplicate_rate])

duplicate_df = pd.DataFrame(
    duplicate_list,
    columns=["데이터", "중복 행 수", "중복률 (%)"],
)

duplicate_df["중복률 (%)"] = duplicate_df["중복률 (%)"].round(2)

duplicate_df

,데이터,중복 행 수,중복률 (%)
0,trade_area,0,0.0
1,stores,0,0.0
2,sales,0,0.0
3,floating_population,0,0.0
4,resident_population,0,0.0
5,working_population,0,0.0
6,facilities,0,0.0


## 4. 데이터 유효성 탐색

통계적으로 드문 값을 임의로 제거하지 않고, 데이터 구조와 의미가 명확한 최소 조건을 검사한다.

### 4.1 수치형 기술통계

숫자로 읽힌 코드 컬럼을 제외하고 수치형 지표의 분포를 확인한다. `min`과 `max`로 범위를 보고, `mean`과 `50%`의 차이로 치우침을 살펴본다. 이 표만으로 이상값을 확정하지는 않는다.

In [9]:
for name, df in datasets.items():
    numeric_columns = [
        column
        for column in df.select_dtypes(include="number").columns
        if "코드" not in column
    ]

    print(f"[{name}]")
    display(
        df[numeric_columns]
        .describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99])
        .T
    )

[trade_area]


,count,mean,std,min,1%,25%,50%,75%,99%,max
X_좌표_EPSG5181,1650.0,198981.845455,7280.720618,182509.0,183944.39,192815.25,200096.0,204366.25,212960.39,215352.0
Y_좌표_EPSG5181,1650.0,449875.451515,5590.503325,437249.0,439431.66,445178.50,449849.5,453563.50,462961.78,465573.0
상권_영역_면적_제곱미터,1650.0,99927.952121,118818.942812,1854.0,4026.23,35330.00,71927.5,128147.25,471762.94,2462734.0


[stores]


,count,mean,std,min,1%,25%,50%,75%,99%,max
유사업종_점포_수,1604844.0,6.855240,56.184018,0.0,1.0,1.0,2.0,5.0,67.0,12934.0
일반_점포_수,1604844.0,6.313860,55.822311,0.0,0.0,1.0,2.0,5.0,62.0,12925.0
프랜차이즈_점포_수,1604844.0,0.541380,2.402498,0.0,0.0,0.0,0.0,0.0,9.0,151.0
개업_점포_수,1604844.0,0.176664,1.182901,0.0,0.0,0.0,0.0,0.0,3.0,396.0
개업_율,1604844.0,2.485957,11.169777,0.0,0.0,0.0,0.0,0.0,50.0,200.0
폐업_점포_수,1604844.0,0.175380,1.339927,0.0,0.0,0.0,0.0,0.0,3.0,387.0
폐업_율,1604844.0,2.376729,11.727200,0.0,0.0,0.0,0.0,0.0,50.0,500.0
원본_기준_일자,1604844.0,20233.868378,15.086125,20211.0,20211.0,20222.0,20233.0,20244.0,20261.0,20261.0


[sales]


...

,count,mean,std,min,1%,25%,50%,75%,99%,max
총_상주인구_수,34275.0,2322.584041,2280.774480,1.0,7.0,640.0,1617.0,3271.0,9998.00,21341.0
남성_상주인구_수,34275.0,1138.400963,1118.233946,0.0,4.0,319.0,800.0,1607.0,4932.00,10459.0
여성_상주인구_수,34275.0,1184.183078,1170.422729,0.0,2.0,322.0,822.0,1685.0,5138.00,10882.0
10대_상주인구_수,34275.0,224.611466,254.320815,0.0,0.0,51.0,144.0,312.0,1139.00,3078.0
20대_상주인구_수,34275.0,388.425791,451.338135,0.0,0.0,89.0,251.0,529.0,2089.00,5174.0
30대_상주인구_수,34275.0,400.805748,442.821182,0.0,0.0,97.0,265.0,557.0,2181.00,4207.0
40대_상주인구_수,34275.0,335.852691,349.687835,0.0,1.0,86.0,226.0,468.0,1566.00,4280.0
50대_상주인구_수,34275.0,360.389292,360.825104,0.0,1.0,95.0,249.0,516.0,1570.56,3161.0
60대_이상_상주인구_수,34275.0,612.499052,610.413556,0.0,2.0,165.0,427.0,862.0,2742.00,5181.0
남성_10대_상주인구_수,34275.0,115.025996,130.666794,0.0,0.0,26.0,74.0,160.0,591.00,1586.0


[working_population]


,count,mean,std,min,1%,25%,50%,75%,99%,max
총_직장인구_수,34386.0,2719.904961,10431.143442,1.0,11.0,188.0,461.0,1261.0,48521.0,214604.0
남성_직장인구_수,34386.0,1662.289798,6491.023931,0.0,6.0,107.0,269.0,739.0,29616.0,123445.0
여성_직장인구_수,34386.0,1057.615163,4016.414319,0.0,3.0,76.0,185.0,503.0,18485.0,91159.0
10대_직장인구_수,34386.0,6.330512,38.557341,0.0,0.0,0.0,0.0,0.0,170.0,946.0
20대_직장인구_수,34386.0,492.168412,2024.591571,0.0,0.0,19.0,60.0,193.0,9726.0,43270.0
30대_직장인구_수,34386.0,781.080352,3305.607712,0.0,1.0,38.0,105.0,314.0,14665.0,71925.0
40대_직장인구_수,34386.0,694.679114,2706.820061,0.0,3.0,46.0,115.0,314.0,13020.0,55534.0
50대_직장인구_수,34386.0,513.066859,1810.692292,0.0,3.0,43.0,109.0,278.0,8196.0,34108.0
60대_이상_직장인구_수,34386.0,232.579713,684.375201,0.0,1.0,25.0,62.0,160.0,2842.0,12788.0
남성_10대_직장인구_수,34386.0,2.536992,16.206749,0.0,0.0,0.0,0.0,0.0,67.0,348.0


[facilities]


,count,mean,std,min,1%,25%,50%,75%,99%,max
총_집객시설_수,33138.0,20.871356,32.260254,1.0,1.0,5.0,12.0,23.0,158.0,594.0
관공서_수,33138.0,0.695184,1.171712,0.0,0.0,0.0,0.0,1.0,5.0,14.0
은행_수,33138.0,1.014575,3.009396,0.0,0.0,0.0,0.0,1.0,13.0,57.0
종합병원_수,33138.0,0.015209,0.127459,0.0,0.0,0.0,0.0,0.0,1.0,2.0
일반병원_수,33138.0,0.107098,0.395562,0.0,0.0,0.0,0.0,0.0,2.0,4.0
약국_수,33138.0,2.359316,4.033607,0.0,0.0,0.0,1.0,3.0,20.0,52.0
유치원_수,33138.0,0.111534,0.385393,0.0,0.0,0.0,0.0,0.0,2.0,6.0
초등학교_수,33138.0,0.008238,0.090392,0.0,0.0,0.0,0.0,0.0,0.0,1.0
중학교_수,33138.0,0.002535,0.050284,0.0,0.0,0.0,0.0,0.0,0.0,1.0
고등학교_수,33138.0,0.008238,0.109421,0.0,0.0,0.0,0.0,0.0,0.0,3.0


### 4.2 핵심 키 고유성

각 데이터의 분석 단위를 나타내는 핵심 키 조합이 한 행씩만 존재하는지 확인한다. `전체 행 수`와 `고유 키 수`가 같고 `중복 키 수`가 0이면 정상이다.

In [10]:
KEY_COLUMNS = {
    "trade_area": ["상권_코드"],
    "stores": ["기준_년분기_코드", "상권_코드", "서비스_업종_코드"],
    "sales": ["기준_년분기_코드", "상권_코드", "서비스_업종_코드"],
    "floating_population": ["기준_년분기_코드", "상권_코드"],
    "resident_population": ["기준_년분기_코드", "상권_코드"],
    "working_population": ["기준_년분기_코드", "상권_코드"],
    "facilities": ["기준_년분기_코드", "상권_코드"],
}

key_check_list = []

for name, key_columns in KEY_COLUMNS.items():
    df = datasets[name]
    unique_key_count = df[key_columns].drop_duplicates().shape[0]
    duplicate_key_count = df.duplicated(subset=key_columns).sum()

    key_check_list.append([
        name,
        " + ".join(key_columns),
        len(df),
        unique_key_count,
        duplicate_key_count,
    ])

key_check_df = pd.DataFrame(
    key_check_list,
    columns=["데이터", "핵심 키", "전체 행 수", "고유 키 수", "중복 키 수"],
)

key_check_df

,데이터,핵심 키,전체 행 수,고유 키 수,중복 키 수
0,trade_area,상권_코드,1650,1650,0
1,stores,기준_년분기_코드 + 상권_코드 + 서비스_업종_코드,1604844,1604844,0
2,sales,기준_년분기_코드 + 상권_코드 + 서비스_업종_코드,460329,460329,0
3,floating_population,기준_년분기_코드 + 상권_코드,34633,34633,0
4,resident_population,기준_년분기_코드 + 상권_코드,34275,34275,0
5,working_population,기준_년분기_코드 + 상권_코드,34386,34386,0
6,facilities,기준_년분기_코드 + 상권_코드,33138,33138,0


### 4.3 상권 코드 참조 확인

점포·매출·인구·시설 데이터의 상권 코드가 상권 마스터인 `trade_area`에 존재하는지 확인한다. `미등록 상권 코드 수`와 `미등록 행 수`가 모두 0이면 정상이다. 데이터별 상권 코드 수 자체는 수집 기간과 제공 범위에 따라 다를 수 있다.

In [11]:
trade_area_codes = set(datasets["trade_area"]["상권_코드"])
reference_list = []

for name, df in datasets.items():
    if name == "trade_area":
        continue

    not_in_master = ~df["상권_코드"].isin(trade_area_codes)

    reference_list.append([
        name,
        df["상권_코드"].nunique(),
        df.loc[not_in_master, "상권_코드"].nunique(),
        not_in_master.sum(),
    ])

reference_df = pd.DataFrame(
    reference_list,
    columns=["데이터", "상권 코드 수", "미등록 상권 코드 수", "미등록 행 수"],
)

reference_df

,데이터,상권 코드 수,미등록 상권 코드 수,미등록 행 수
0,stores,1650,0,0
1,sales,1596,0,0
2,floating_population,1650,0,0
3,resident_population,1637,0,0
4,working_population,1643,0,0
5,facilities,1578,0,0


### 4.4 코드와 명칭의 일관성

하나의 코드에 서로 다른 명칭이 연결되어 있는지 확인한다. `여러 명칭이 연결된 코드 수`가 0이면 정상이다. 0보다 크면 명칭 변경, 오탈자 또는 코드 매핑 문제인지 확인한다.

In [12]:
CODE_NAME_PAIRS = [
    ("상권_코드", "상권_코드명"),
    ("상권_구분_코드", "상권_구분_코드명"),
    ("서비스_업종_코드", "서비스_업종_코드명"),
    ("자치구_코드", "자치구_명"),
    ("행정동_코드", "행정동_명"),
]

code_name_list = []

for name, df in datasets.items():
    for code_column, name_column in CODE_NAME_PAIRS:
        if code_column in df.columns and name_column in df.columns:
            name_count = df.groupby(code_column)[name_column].nunique()
            inconsistent_count = (name_count > 1).sum()

            code_name_list.append([
                name,
                code_column,
                name_column,
                inconsistent_count,
            ])

code_name_df = pd.DataFrame(
    code_name_list,
    columns=["데이터", "코드 컬럼", "명칭 컬럼", "여러 명칭이 연결된 코드 수"],
)

code_name_df

,데이터,코드 컬럼,명칭 컬럼,여러 명칭이 연결된 코드 수
0,trade_area,상권_코드,상권_코드명,0
1,trade_area,상권_구분_코드,상권_구분_코드명,0
2,trade_area,자치구_코드,자치구_명,0
3,trade_area,행정동_코드,행정동_명,0
4,stores,상권_코드,상권_코드명,2
5,stores,상권_구분_코드,상권_구분_코드명,0
6,stores,서비스_업종_코드,서비스_업종_코드명,0
7,sales,상권_코드,상권_코드명,2
8,sales,상권_구분_코드,상권_구분_코드명,0
9,sales,서비스_업종_코드,서비스_업종_코드명,0


### 4.5 수치형 값의 최소 조건

금액·인구·점포·시설 수는 `음수 개수`와 `무한대 개수`가 0이어야 한다. 건수형 컬럼은 `소수 개수`도 0이어야 한다. 비율은 0~100, 면적은 0 초과를 정상 범위로 본다. `0 개수`는 참고값이며 매출·점포·시설에서 0은 정상일 수 있으므로 오류로 판단하지 않는다.

In [13]:
numeric_check_list = []

for name, df in datasets.items():
    numeric_columns = [
        column
        for column in df.select_dtypes(include="number").columns
        if "코드" not in column
    ]

    for column in numeric_columns:
        values = df[column]
        negative_count = (values < 0).sum()
        zero_count = (values == 0).sum()
        infinity_count = values.isin([float("inf"), float("-inf")]).sum()

        decimal_count = 0
        if column.endswith("_수") or column.endswith("_건수"):
            decimal_count = ((values.dropna() % 1) != 0).sum()

        range_error_count = 0
        if column.endswith("_율"):
            range_error_count = ((values < 0) | (values > 100)).sum()
        elif "면적" in column:
            range_error_count = (values <= 0).sum()

        numeric_check_list.append([
            name, column, values.min(), values.max(),
            negative_count, zero_count, decimal_count,
            infinity_count, range_error_count,
        ])

numeric_check_df = pd.DataFrame(
    numeric_check_list,
    columns=[
        "데이터", "컬럼", "최솟값", "최댓값", "음수 개수",
        "0 개수", "소수 개수", "무한대 개수", "범위 오류 개수",
    ],
)

numeric_check_df

,데이터,컬럼,최솟값,최댓값,음수 개수,0 개수,소수 개수,무한대 개수,범위 오류 개수
0,trade_area,X_좌표_EPSG5181,182509.0,215352.0,0,0,0,0,0
1,trade_area,Y_좌표_EPSG5181,437249.0,465573.0,0,0,0,0,0
2,trade_area,상권_영역_면적_제곱미터,1854.0,2462734.0,0,0,0,0,0
3,stores,유사업종_점포_수,0.0,12934.0,0,11095,0,0,0
4,stores,일반_점포_수,0.0,12925.0,0,56092,0,0,0
...,...,...,...,...,...,...,...,...,...
146,facilities,철도역_수,0.0,0.0,0,33138,0,0,0
147,facilities,버스터미널_수,0.0,1.0,0,33096,0,0,0
148,facilities,지하철역_수,0.0,5.0,0,29043,0,0,0
149,facilities,버스정류장_수,0.0,85.0,0,6468,0,0,0


### 4.6 분기 코드 형식

분기 코드가 연도 4자리와 분기 1자리로 구성됐는지 확인한다. 예를 들어 `20241`은 2024년 1분기를 의미한다. 마지막 자리는 1~4만 허용하며 `형식 오류 수`가 0이면 정상이다.

In [14]:
quarter_list = []

for name, df in datasets.items():
    if "기준_년분기_코드" not in df.columns:
        continue

    quarter = df["기준_년분기_코드"].astype("string")
    invalid_count = (~quarter.str.fullmatch(r"\d{4}[1-4]", na=False)).sum()

    quarter_list.append([
        name,
        quarter.nunique(),
        quarter.min(),
        quarter.max(),
        invalid_count,
    ])

quarter_df = pd.DataFrame(
    quarter_list,
    columns=["데이터", "분기 수", "최초 분기", "최근 분기", "형식 오류 수"],
)

quarter_df

,데이터,분기 수,최초 분기,최근 분기,형식 오류 수
0,stores,21,20211,20261,0
1,sales,21,20211,20261,0
2,floating_population,21,20211,20261,0
3,resident_population,21,20211,20261,0
4,working_population,21,20211,20261,0
5,facilities,21,20211,20261,0


### 4.7 주요 총계와 구성값의 일치

총계가 주요 구성값의 합과 일치하는지 확인한다. 원천 데이터의 반올림을 고려해 절대 차이가 1 이하이면 일치로 판단한다. `불일치 행 수`가 0이면 정상이다. 세부 항목이 전체 구성을 완전히 포함한다고 확신할 수 없는 시설 데이터는 이 검사에서 제외한다.

In [15]:
SUM_RULES = {
    "stores": [
        ("유사업종 점포 수", "유사업종_점포_수", ["일반_점포_수", "프랜차이즈_점포_수"]),
    ],
    "sales": [
        ("분기 매출 금액", "분기_매출_금액", ["주중_매출_금액", "주말_매출_금액"]),
        ("분기 매출 건수", "분기_매출_건수", ["주중_매출_건수", "주말_매출_건수"]),
    ],
    "floating_population": [
        ("총 유동인구 수", "총_유동인구_수", ["남성_유동인구_수", "여성_유동인구_수"]),
    ],
    "resident_population": [
        ("총 상주인구 수", "총_상주인구_수", ["남성_상주인구_수", "여성_상주인구_수"]),
        ("총 가구 수", "총_가구_수", ["아파트_가구_수", "비아파트_가구_수"]),
    ],
    "working_population": [
        ("총 직장인구 수", "총_직장인구_수", ["남성_직장인구_수", "여성_직장인구_수"]),
    ],
}

sum_check_list = []

for name, rules in SUM_RULES.items():
    df = datasets[name]

    for check_name, total_column, part_columns in rules:
        part_sum = df[part_columns].sum(axis=1, min_count=len(part_columns))
        difference = (df[total_column] - part_sum).abs()
        mismatch_count = (difference > 1).sum()

        sum_check_list.append([
            name, check_name, mismatch_count, difference.max(),
        ])

sum_check_df = pd.DataFrame(
    sum_check_list,
    columns=["데이터", "검사 항목", "불일치 행 수", "최대 차이"],
)

sum_check_df

,데이터,검사 항목,불일치 행 수,최대 차이
0,stores,유사업종 점포 수,0,0.0
1,sales,분기 매출 금액,0,0.0
2,sales,분기 매출 건수,0,0.0
3,floating_population,총 유동인구 수,2610,3.0
4,resident_population,총 상주인구 수,0,0.0
5,resident_population,총 가구 수,0,0.0
6,working_population,총 직장인구 수,0,0.0


## 5. 데이터 유효성 검사 결과 해석

| 항목 | 판단 | 분석 결과 |
|---|---|---|
| 4.1 수치형 기술통계 | 참고 | 수치형 지표에서 음수·무한대·소수형 건수는 발견되지 않았다. 매출·인구·시설 데이터에는 0이 다수 존재하지만, 영업이나 시설이 없는 경우를 나타낼 수 있으므로 0 자체를 오류로 판단하지 않는다. 기술통계의 큰 최댓값은 실제 대형 상권일 수 있어 별도 아웃라이어로 제거하지 않는다. |
| 4.2 핵심 키 고유성 | 정상 | 7개 데이터 모두 전체 행 수와 고유 핵심 키 조합 수가 같고 중복 키 수가 0이다. 따라서 현재 정의한 분석 단위에서 중복 레코드는 없다. |
| 4.3 상권 코드 참조 | 정상 | 모든 데이터의 상권 코드가 `trade_area`에 등록되어 있으며 미등록 상권 코드와 미등록 행은 0건이다. 데이터별 상권 수 차이는 제공 범위 차이로 볼 수 있다. |
| 4.4 코드와 명칭의 일관성 | 검토 필요 | 시계열 데이터에서 상권 코드 2개가 서로 다른 명칭을 사용한다. `3110024`는 `혜화동주민센터`와 `혜회동주민센터`, `3110379`는 `KT&G 북부지사`와 `KTNG 북부지사`가 함께 존재한다. 시기별 명칭 변경 또는 표기 차이로 보이며, 분석용 명칭은 `trade_area` 기준으로 통일할지 결정할 필요가 있다. |
| 4.5 수치형 최소 조건 | 대체로 정상 | 모든 데이터에서 음수·무한대·계수형 소수값은 0건이다. 다만 `stores`에서 개업률 100 초과가 14건(최대 200), 폐업률 100 초과가 767건(최대 500) 확인됐다. 공식 데이터 설명은 두 컬럼을 개업률·폐업률 수치로 제공하지만 상한을 명시하지 않으므로, 현재의 0~100 기준은 확정 오류가 아니라 검토 기준으로 해석한다. |
| 4.6 분기 코드 형식 | 정상 | 분기형 데이터 6종 모두 2021년 1분기(`20211`)부터 2026년 1분기(`20261`)까지 21개 분기를 포함하며 형식 오류는 0건이다. |
| 4.7 총계와 구성값 일치 | 대체로 정상 | 점포, 매출, 상주인구, 가구, 직장인구는 모두 허용 오차 안에서 일치한다. 유동인구는 2,610행이 현재 허용 오차 1을 넘지만 최대 차이가 3명에 불과하므로 집계·반올림 차이로 해석할 수 있다. 유동인구에 한해 허용 오차를 3으로 조정하는 것이 적절하다. |

### 종합 판단

핵심 키, 상권 코드 참조, 분기 형식과 주요 합계 관계는 전반적으로 정상이다. 현재 즉시 제거해야 할 명확한 오류값은 발견되지 않았다. 후속 작업에서는 상권명 2개 코드의 표기 통일 여부를 정하고, 개·폐업률 100 초과 값을 원천 산식에 따라 해석하며, 유동인구 합계 검사 허용 오차를 3으로 조정하는 것이 좋다.

참고: [서울시 상권분석서비스 점포-상권 데이터 설명](https://data.seoul.go.kr/dataList/OA-15577/S/1/datasetView.do?tab=A)

## 6. 검토 항목 추가 탐색

### 6.1 개업률·폐업률 100% 초과

100을 초과한 비율의 건수와 최댓값을 확인하고, 해당 행의 전체 점포 수와 개·폐업 점포 수를 함께 살펴본다. 전체 점포 수보다 한 분기 동안 개업하거나 폐업한 점포 수가 많다면 비율은 100을 초과할 수 있다.

In [16]:
stores = datasets["stores"]

rate_summary = pd.DataFrame({
    "항목": ["개업률", "폐업률"],
    "100 초과 행 수": [
        (stores["개업_율"] > 100).sum(),
        (stores["폐업_율"] > 100).sum(),
    ],
    "최댓값": [stores["개업_율"].max(), stores["폐업_율"].max()],
})

display(rate_summary)

rate_review = stores.loc[
    (stores["개업_율"] > 100) | (stores["폐업_율"] > 100),
    [
        "기준_년분기_코드", "상권_코드명", "서비스_업종_코드명",
        "유사업종_점포_수", "개업_점포_수", "개업_율",
        "폐업_점포_수", "폐업_율",
    ],
]

rate_review.sort_values(
    ["폐업_율", "개업_율"],
    ascending=False,
).head(30)

,항목,100 초과 행 수,최댓값
0,개업률,14,200.0
1,폐업률,767,500.0


,기준_년분기_코드,상권_코드명,서비스_업종_코드명,유사업종_점포_수,개업_점포_수,개업_율,폐업_점포_수,폐업_율
1111218,20223,이수역 10번,자동차수리,1.0,0.0,0.0,5.0,500.0
350826,20251,염창무학아파트,전자상거래업,2.0,0.0,0.0,9.0,450.0
255567,20252,중림동,스포츠 강습,1.0,0.0,0.0,4.0,400.0
308066,20251,송화벽화시장(송화골목시장),전자상거래업,1.0,0.0,0.0,4.0,400.0
889407,20232,까치산역 3번,PC방,1.0,0.0,0.0,4.0,400.0
991203,20231,한강진역 3번,예술학원,1.0,0.0,0.0,4.0,400.0
6927,20261,장위전통시장,미곡판매,1.0,1.0,100.0,3.0,300.0
117460,20254,대림1동주민센터,치킨전문점,1.0,1.0,100.0,3.0,300.0
129390,20254,경의중앙 신촌역,편의점,1.0,1.0,100.0,3.0,300.0
401699,20244,등촌역,가전제품수리,1.0,1.0,100.0,3.0,300.0


### 6.2 유동인구 성별 합계 차이

`총 유동인구 - 남성 유동인구 - 여성 유동인구`를 계산해 차이의 전체 분포를 확인한다. 차이가 작은 정수 범위에만 집중되어 있으면 개별 성별 값의 반올림 과정에서 발생한 차이로 볼 수 있다.

In [17]:
floating = datasets["floating_population"]

floating_difference = (
    floating["총_유동인구_수"]
    - floating["남성_유동인구_수"]
    - floating["여성_유동인구_수"]
)

floating_difference_df = (
    floating_difference.value_counts()
    .sort_index()
    .rename_axis("합계 차이")
    .reset_index(name="행 수")
)

floating_difference_df

,합계 차이,행 수
0,-3.0,59
1,-2.0,1304
2,-1.0,7610
3,0.0,17167
4,1.0,7246
5,2.0,1184
6,3.0,63


### 6.3 상권명 변경 이력

여러 명칭이 연결된 상권 코드 2개의 분기별 명칭을 확인한다. 특정 분기를 기준으로 모든 데이터에서 동일하게 명칭이 바뀌었다면 무작위 오류보다는 원천의 명칭 정비 또는 변경으로 해석할 수 있다.

In [18]:
review_codes = [3110024, 3110379]
name_history_list = []

for name, df in datasets.items():
    if "기준_년분기_코드" not in df.columns:
        continue

    history = df.loc[
        df["상권_코드"].isin(review_codes),
        ["기준_년분기_코드", "상권_코드", "상권_코드명"],
    ].drop_duplicates()

    history.insert(0, "데이터", name)
    name_history_list.append(history)

name_history_df = pd.concat(name_history_list, ignore_index=True)

name_history_df.sort_values(
    ["상권_코드", "기준_년분기_코드", "데이터"]
)

,데이터,기준_년분기_코드,상권_코드,상권_코드명
212,facilities,20211,3110024,혜회동주민센터
98,floating_population,20211,3110024,혜회동주민센터
141,resident_population,20211,3110024,혜회동주민센터
71,sales,20211,3110024,혜회동주민센터
41,stores,20211,3110024,혜회동주민센터
...,...,...,...,...
248,facilities,20261,3110379,KTNG 북부지사
122,floating_population,20261,3110379,KTNG 북부지사
161,resident_population,20261,3110379,KTNG 북부지사
0,stores,20261,3110379,KTNG 북부지사


### 추가 탐색 결과

- **개·폐업률:** 개업률 100 초과는 14행, 폐업률 100 초과는 767행이다. 제공 기관에서 실제 산식을 제공하고 있지 않아서 정확한 판단을 할 수 없지만, 기준 분기의 점포수를 분모로 계산하는 것으로 추정된다. 또한 실제 연구자료나 언론보도에서 폐업률을 100% 초과한 수치를 언급하는 것으로 보아 우선 개업률과 폐업률을 이상치로 언급만 하고 우리 분석에서는 그대로 사용하기로 한다.
- **유동인구 합계:** 총계와 세부 합계 사이에 작은 차이가 확인되지만, 공식 문서에는 반올림 방식이나 합계 일치 조건이 명시되어 있지 않다. 그러므로 이를 단순 반올림으로 단정하지 않고 원천 추정·공간배분·집계 과정에서 발생한 미세한 집계 잔차로 간주하는 것이 적절하며 오차율이 매우 낮기 때문에 데이터를 그대로 사용하기로 한다.
- **상권명:** "두 상권 모두 2024년 4분기까지 과거 명칭을 사용하고 2025년 1분기부터 새 명칭으로 일괄 변경됐다. 이전 명칭은 "혜회동주민센터"에서 "혜화동주민센터"로 KT&G 북부지사에서 KTNG 북부지사로 변경되었으며 오탈자 및 명칭 통합으로 인한 변경으로 추정된다.따라서 명칭은 최신 `trade_area` 명칭을 사용하는 것이 적절하다.